In [16]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime

pd.set_option('display.max_rows', 500)


In [17]:
# ── 1. PATH CONFIGURATION ───────────────
BASE_DATA_DIR = "../../../data"

RAW_DIR    = os.path.join(BASE_DATA_DIR, "raw")
BRONZE_DIR = os.path.join(BASE_DATA_DIR, "bronze")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(BRONZE_DIR, exist_ok=True)

CSV_URL = "https://www.data.gouv.fr/api/1/datasets/r/6fd17a6c-519b-465c-a7fd-ad2955fafc76"

path_csv_raw         = os.path.join(RAW_DIR, "dictionnaire des nuances politiques.csv")
path_communes_bronze = os.path.join(BRONZE_DIR, "bronze_dictionnaire_des_nuances_politiques.csv")


In [18]:
# ── 2. RAW LAYER: Landing Zone ──────────────────────────────────────────────
# We keep the source file exactly as it is.
if not os.path.exists(path_csv_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(CSV_URL, timeout=180)
    resp.raise_for_status()
    with open(path_csv_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_csv_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


✅ Raw file already exists in ../../../data/raw


In [19]:
# ── 3. BRONZE LAYER: Ingestion & Metadata ───────────────────────────────────
# Bronze is for "raw-ish" data in a readable format with audit columns.
print("\n🛠 Processing RAW -> BRONZE...")
df_bronze = pd.read_csv(path_csv_raw, sep=None, engine="python", dtype=str)

# Normalize column names by trimming surrounding spaces
df_bronze.columns = [c.strip() if isinstance(c, str) else c for c in df_bronze.columns]
df_bronze['extraction_source_url'] = CSV_URL
df_bronze['ingestion_timestamp']   = datetime.now().isoformat()
df_bronze['source_file_name']      = os.path.basename(path_csv_raw)




🛠 Processing RAW -> BRONZE...


In [20]:
display(df_bronze.head())

,Nuance,Libellé,extraction_source_url,ingestion_timestamp,source_file_name
0,EXG,Extrême gauche,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
1,COM,Communiste,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
2,PG,Parti de Gauche,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
3,SOC,Socialiste,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
4,RDG,Radical de Gauche,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv


In [21]:
# --- Ajout manuel de nuances (sans modifier le reste du pipeline) ---

nouvelles_nuances = pd.DataFrame([
    {"Nuance": "RES", "Libellé": "Résistons !"},
    {"Nuance": "MODEM", "Libellé": "Mouvement démocrate"},
    {"Nuance": "EELV", "Libellé": "Europe Écologie Les Verts"}
])

# Ajouter les métadonnées demandées
nouvelles_nuances["extraction_source_url"] = "manual_correction"
nouvelles_nuances["ingestion_timestamp"] = datetime.now().isoformat()
nouvelles_nuances["source_file_name"] = "manual_correction"

# Ajouter au dataset principal
df_bronze = pd.concat([df_bronze, nouvelles_nuances], ignore_index=True)
display(df_bronze)

,Nuance,Libellé,extraction_source_url,ingestion_timestamp,source_file_name
0,EXG,Extrême gauche,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
1,COM,Communiste,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
2,PG,Parti de Gauche,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
3,SOC,Socialiste,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
4,RDG,Radical de Gauche,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
5,DVG,Divers gauche,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
6,VEC,Europe Ecologie / Les Verts,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
7,ECO,Ecologiste,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
8,REG,Régionaliste,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv
9,AUT,Autres,https://www.data.gouv.fr/api/1/datasets/r/6fd1...,2026-03-28T22:27:49.891940,dictionnaire des nuances politiques.csv


In [22]:
df_bronze.to_csv(path_communes_bronze, index=False, sep=";", encoding="utf-8")

print(f"✅ BRONZE dataset created: {path_communes_bronze}")
print(f"📊 Rows: {len(df_bronze)} | Columns: {len(df_bronze.columns)}")


✅ BRONZE dataset created: ../../../data/bronze/bronze_dictionnaire_des_nuances_politiques.csv
📊 Rows: 89 | Columns: 5
